<a href="https://colab.research.google.com/github/vv2281/MAT-422/blob/main/HW3VV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_digits
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(422)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

#3.7.1 Mathematical Formulation

A feedforward network has layers l = 0, 1, …, L with nˡ neurons each. Layer 0 is the input, a⁰ = x.

Each layer l = 1, …, L applies an affine map followed by an activation function σ, applied entrywise: zˡ = Wˡ aˡ⁻¹ + bˡ        (weighted input), and aˡ = σ(zˡ) (activation)

-Wˡ is an nˡ × nˡ⁻¹ weight matrix. Its entry wˡⱼₖ is the weight from neuron k in layer l − 1 to neuron j in layer l.
-bˡ ∈ R^(nˡ) is the bias vector.

The network is the composition f(x) = aᴸ = σ(Wᴸ σ( … σ(W¹x + b¹) … ) + bᴸ). It has ∑ₗ nˡ(nˡ⁻¹ + 1) parameters.

For a batch of m inputs, stack them as the columns of a matrix A⁰ (n⁰ × m). Then Zˡ = WˡAˡ⁻¹ + bˡ, with bˡ added to every column.

A single neuron with sigmoid activation, a = σ(wᵀx + b), is exactly the logistic regression model from Section 3.4.

##Example 1: Forward pass of a 2-3-1 network

Network sizes [2, 3, 1]: 2 inputs, one hidden layer with 3 neurons, 1 output. The forward pass is computed one input at a time with the formulas above, then for the whole batch at once, and the two results are compared.

In [ ]:
def init_params(sizes, rng):
    return [(rng.normal(0, np.sqrt(1 / sizes[l]), size=(sizes[l + 1], sizes[l])), np.zeros((sizes[l + 1], 1)))
            for l in range(len(sizes) - 1)]

sigmoid = lambda z: 1 / (1 + np.exp(-np.clip(z, -500, 500)))

sizes = [2, 3, 1]
params = init_params(sizes, rng)
X = rng.normal(size=(2, 5))

single = []
for i in range(X.shape[1]):
    a = X[:, [i]]
    for W, b in params:
        a = sigmoid(W @ a + b)
    single.append(a.item())

A = X
for l, (W, b) in enumerate(params, start=1):
    Z = W @ A + b
    A = sigmoid(Z)
    print(f"layer {l}: W {W.shape}, b {b.shape}, Z and A {Z.shape}")

print("number of parameters:", sum(W.size + b.size for W, b in params), "=", sum(sizes[l + 1] * (sizes[l] + 1) for l in range(len(sizes) - 1)))
print("one at a time:", np.round(single, 6))
print("batch        :", np.round(A.ravel(), 6))

fig, ax = plt.subplots(figsize=(6, 4))
pos = {(l, j): (l, j - (n - 1) / 2) for l, n in enumerate(sizes) for j in range(n)}
for l in range(len(sizes) - 1):
    W = params[l][0]
    for j in range(sizes[l + 1]):
        for k in range(sizes[l]):
            (x0, y0), (x1, y1) = pos[(l, k)], pos[(l + 1, j)]
            ax.plot([x0, x1], [y0, y1], color="tab:red" if W[j, k] > 0 else "tab:blue", lw=1 + 2 * abs(W[j, k]))
for (l, j), (x, y) in pos.items():
    ax.add_patch(plt.Circle((x, y), 0.15, color="white", ec="k", zorder=3))
ax.set_xticks(range(len(sizes))); ax.set_xticklabels(["input", "hidden", "output"]); ax.set_yticks([])
ax.set_aspect("equal"); ax.set_title("2-3-1 network (red = positive weight, blue = negative, width = |w|)")
plt.show()

#3.7.2 Activation Functions
For multi-class output, the softmax activation turns a vector z into probabilities: softmax(z)ⱼ = e^(zⱼ) / ∑ₖ e^(zₖ)

Why the activation must be nonlinear: without σ, two layers collapse into one,

 W²(W¹x + b¹) + b² = (W²W¹)x + (W²b¹ + b²)

so a deep linear network is still just a linear function.

Vanishing gradients: sigmoid' ≤ ¼ and tanh' ≤ 1, so the gradient picks up one σ' factor per layer and can shrink quickly in deep networks. ReLU has derivative 1 wherever z > 0.

In [ ]:
acts = {
    "sigmoid": (sigmoid, lambda z: sigmoid(z) * (1 - sigmoid(z))),
    "tanh": (np.tanh, lambda z: 1 - np.tanh(z) ** 2),
    "relu": (lambda z: np.maximum(0, z), lambda z: (z > 0).astype(float)),
    "leaky_relu": (lambda z: np.where(z > 0, z, 0.01 * z), lambda z: np.where(z > 0, 1.0, 0.01)),
}

def softmax(Z):
    E = np.exp(Z - Z.max(axis=0, keepdims=True))
    return E / E.sum(axis=0, keepdims=True)

z = np.linspace(-4, 4, 401)
h = 1e-6
for name, (f, df) in acts.items():
    zz = z[np.abs(z) > 1e-3]
    print(f"{name:11s} max |σ' formula − finite difference| = {np.abs(df(zz) - (f(zz + h) - f(zz - h)) / (2 * h)).max():.2e}")

zs = np.array([[2.0], [1.0], [0.1]])
print("softmax([2, 1, 0.1]) =", np.round(softmax(zs).ravel(), 4), " sum =", softmax(zs).sum())

(W1, b1), (W2, b2) = init_params([2, 3, 1], rng)
xt = rng.normal(size=(2, 4))
print("linear 2-layer = single linear layer:", np.allclose(W2 @ (W1 @ xt + b1) + b2, (W2 @ W1) @ xt + (W2 @ b1 + b2)))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, (f, df) in acts.items():
    ax[0].plot(z, f(z), label=name)
    ax[1].plot(z, df(z), label=name)
ax[0].set_ylim(-1.5, 4); ax[0].set_title("σ(z)"); ax[0].legend()
ax[1].set_title("σ'(z)"); ax[1].legend()
plt.tight_layout(); plt.show()

#3.7.3 Cost Function

Training finds the parameters that minimize an average cost over the n training pairs (xᵢ, yᵢ): C = (1/n) ∑ᵢ Cᵢ

Two common choices:
Quadratic (MSE):   Cᵢ = ½ ‖aᴸ(xᵢ) − yᵢ‖²
Cross-entropy:     Cᵢ = −∑ⱼ yᵢⱼ log aᴸⱼ(xᵢ)        (with softmax or sigmoid output)

What backpropagation needs from the cost is the output error δᴸ = ∂Cᵢ/∂zᴸ:

MSE + sigmoid:          δᴸ = (aᴸ − y) ⊙ σ'(zᴸ)
Cross-entropy + softmax: δᴸ = aᴸ − y

(⊙ is entrywise multiplication.) With MSE, the σ'(zᴸ) factor is close to 0 when the output saturates, even if the prediction is badly wrong, so learning slows down. With cross-entropy that factor cancels.

In [ ]:
zL = np.linspace(-8, 8, 400)
aL = sigmoid(zL)
y = 1.0

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(zL, 0.5 * (aL - y) ** 2, label="MSE ½(a − y)²")
ax[0].plot(zL, -np.log(aL), label="cross-entropy −log a")
ax[0].set_xlabel("zᴸ"); ax[0].set_title("Cost for true label y = 1"); ax[0].legend()
ax[1].plot(zL, np.abs((aL - y) * aL * (1 - aL)), label="MSE: |(a − y)σ'(z)|")
ax[1].plot(zL, np.abs(aL - y), label="cross-entropy: |a − y|")
ax[1].set_xlabel("zᴸ"); ax[1].set_title("|δᴸ| = |∂C/∂zᴸ|  (left side = confidently wrong)"); ax[1].legend()
plt.tight_layout(); plt.show()

On the left side of the plots the network is confidently wrong (a ≈ 0 while y = 1). The cross-entropy gradient is close to 1 there, but the MSE gradient is close to 0, so an MSE-trained network barely moves.

#3.7.4 Backpropagation

Goal: compute ∂C/∂Wˡ and ∂C/∂bˡ for every layer. Define the error of layer l as δˡ = ∂C/∂zˡ. Applying the chain rule to zˡ⁺¹ = Wˡ⁺¹σ(zˡ) + bˡ⁺¹ gives four equations:

(BP1)  δᴸ = ∇ₐC ⊙ σ'(zᴸ)
(BP2)  δˡ = ((Wˡ⁺¹)ᵀ δˡ⁺¹) ⊙ σ'(zˡ)
(BP3)  ∂C/∂bˡ = δˡ
(BP4)  ∂C/∂Wˡ = δˡ (aˡ⁻¹)ᵀ,   i.e.  ∂C/∂wˡⱼₖ = aˡ⁻¹ₖ δˡⱼ

Derivation of BP2: zˡ⁺¹ₖ depends on zˡⱼ only through aˡⱼ = σ(zˡⱼ), so

δˡⱼ = ∑ₖ (∂C/∂zˡ⁺¹ₖ)(∂zˡ⁺¹ₖ/∂zˡⱼ) = ∑ₖ δˡ⁺¹ₖ wˡ⁺¹ₖⱼ σ'(zˡⱼ)

BP3 and BP4 follow from zˡⱼ = ∑ₖ wˡⱼₖ aˡ⁻¹ₖ + bˡⱼ.

The error is computed at the output (BP1) and passed backward one layer at a time (BP2). Each layer costs about the same as in the forward pass, so all the gradients together cost roughly two forward passes. Finite differences would need two forward passes for every single parameter.

For a batch, columns are samples. δˡ becomes a matrix, BP3 sums δˡ over the columns, and BP4 becomes δˡ(Aˡ⁻¹)ᵀ.

In [ ]:
def forward(params, X, act, out):
    f = acts[act][0]
    A, zs, As = X, [], [X]
    for l, (W, b) in enumerate(params):
        Z = W @ A + b
        last = l == len(params) - 1
        A = (softmax(Z) if out == "softmax" else sigmoid(Z)) if last else f(Z)
        zs.append(Z); As.append(A)
    return zs, As

def cost(params, X, Y, act, out, kind):
    AL = forward(params, X, act, out)[1][-1]
    if kind == "mse":
        return 0.5 * np.sum((AL - Y) ** 2) / X.shape[1]
    AL = np.clip(AL, 1e-12, 1 - 1e-12)
    if out == "sigmoid":
        return -np.sum(Y * np.log(AL) + (1 - Y) * np.log(1 - AL)) / X.shape[1]
    return -np.sum(Y * np.log(AL)) / X.shape[1]

def backprop(params, X, Y, act, out, kind):
    df = acts[act][1]
    zs, As = forward(params, X, act, out)
    m = X.shape[1]
    delta = (As[-1] - Y) * sigmoid(zs[-1]) * (1 - sigmoid(zs[-1])) / m if kind == "mse" else (As[-1] - Y) / m
    grads = [None] * len(params)
    for l in range(len(params) - 1, -1, -1):
        grads[l] = (delta @ As[l].T, delta.sum(axis=1, keepdims=True))
        if l > 0:
            delta = (params[l][0].T @ delta) * df(zs[l - 1])
    return grads

##Example 2: Gradient check

Compare backpropagation with centered finite differences for every weight and bias: ∂C/∂θ ≈ [C(θ + h) − C(θ − h)] / (2h)

This is done for a 3-4-5-2 network with tanh hidden layers, once with MSE + sigmoid output (testing BP1 with the σ' factor) and once with cross-entropy + softmax. A relative error near 10⁻⁸ or smaller means backpropagation is correct.

In [ ]:
def grad_check(act, out, kind, h=1e-5):
    p = init_params([3, 4, 5, 2], rng)
    Xg = rng.normal(size=(3, 6))
    Yg = np.eye(2)[:, rng.integers(0, 2, 6)]
    bp = backprop(p, Xg, Yg, act, out, kind)
    num, ana = [], []
    for l in range(len(p)):
        for i in range(2):
            T = p[l][i]
            for idx in np.ndindex(T.shape):
                old = T[idx]
                T[idx] = old + h; cp = cost(p, Xg, Yg, act, out, kind)
                T[idx] = old - h; cm = cost(p, Xg, Yg, act, out, kind)
                T[idx] = old
                num.append((cp - cm) / (2 * h)); ana.append(bp[l][i][idx])
    num, ana = np.array(num), np.array(ana)
    return num, ana, np.linalg.norm(num - ana) / np.linalg.norm(num + ana)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (act, out, kind) in zip(ax, [("tanh", "sigmoid", "mse"), ("tanh", "softmax", "ce")]):
    num, ana, rel = grad_check(act, out, kind)
    print(f"{kind.upper():4s} + {out:7s}: {len(num)} parameters, relative error = {rel:.2e}")
    a.scatter(num, ana, s=12)
    lim = np.abs(num).max() * 1.1
    a.plot([-lim, lim], [-lim, lim], "r--", lw=1)
    a.set_xlabel("finite difference"); a.set_ylabel("backpropagation")
    a.set_title(f"{kind.upper()} + {out}: relative error {rel:.1e}")
plt.tight_layout(); plt.show()

#3.7.5 Backpropagation Algorithm

Training with (mini-batch) stochastic gradient descent:

1. Initialize the weights randomly (scale about 1/√nˡ⁻¹) and the biases to 0. If all weights started equal, every neuron in a layer would receive the same update and they would never become different.
2. For each epoch, shuffle the data and split it into mini-batches of size m. For each mini-batch:
- Forward pass: compute zˡ and aˡ for l = 1, …, L.
- Output error: δᴸ from BP1.
- Backward pass: δˡ for l = L − 1, …, 1 from BP2.
- Gradients: from BP3 and BP4.
- Update: Wˡ ← Wˡ − η ∂C/∂Wˡ and bˡ ← bˡ − η ∂C/∂bˡ, with learning rate η.

C is not convex in the weights (unlike Sections 3.4 and 3.6), so SGD finds a local minimum, and the result depends on the initialization.

In [ ]:
def train(sizes, X, Y, act, out, kind, eta, epochs, batch, seed=0):
    r = np.random.default_rng(seed)
    p = init_params(sizes, r)
    hist = []
    for _ in range(epochs):
        perm = r.permutation(X.shape[1])
        for s in range(0, X.shape[1], batch):
            idx = perm[s:s + batch]
            g = backprop(p, X[:, idx], Y[:, idx], act, out, kind)
            p = [(W - eta * gW, b - eta * gb) for (W, b), (gW, gb) in zip(p, g)]
        hist.append(cost(p, X, Y, act, out, kind))
    return p, np.array(hist)

def predict(p, X, act, out):
    AL = forward(p, X, act, out)[1][-1]
    return (AL[0] >= 0.5).astype(int) if out == "sigmoid" else AL.argmax(axis=0)

##Example 3: XOR (why hidden layers are needed)

(0,0) → 0,   (0,1) → 1,   (1,0) → 1,   (1,1) → 0

No line separates the 1s from the 0s, so a single neuron (logistic regression) can't fit XOR. A network with one hidden layer of 4 tanh neurons can.

In [ ]:
Xx = np.array([[0, 0, 1, 1], [0, 1, 0, 1]], dtype=float)
Yx = np.array([[0, 1, 1, 0]], dtype=float)

p0, h0 = train([2, 1], Xx, Yx, "tanh", "sigmoid", "ce", eta=1.0, epochs=3000, batch=4)
p1, h1 = train([2, 4, 1], Xx, Yx, "tanh", "sigmoid", "ce", eta=1.0, epochs=3000, batch=4)
print("no hidden layer  outputs:", np.round(forward(p0, Xx, "tanh", "sigmoid")[1][-1].ravel(), 3))
print("one hidden layer outputs:", np.round(forward(p1, Xx, "tanh", "sigmoid")[1][-1].ravel(), 3))

g = np.linspace(-0.5, 1.5, 200)
G1, G2 = np.meshgrid(g, g)
grid = np.vstack([G1.ravel(), G2.ravel()])
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].semilogy(h0, label="2-1 (no hidden layer)")
ax[0].semilogy(h1, label="2-4-1")
ax[0].set_xlabel("epoch"); ax[0].set_title("Cross-entropy cost"); ax[0].legend()
for a, p, t in [(ax[1], p0, "2-1: no hidden layer"), (ax[2], p1, "2-4-1: one hidden layer")]:
    P = forward(p, grid, "tanh", "sigmoid")[1][-1].reshape(G1.shape)
    a.contourf(G1, G2, P, levels=20, cmap="RdBu_r", alpha=0.6)
    a.contour(G1, G2, P, levels=[0.5], colors="k")
    a.scatter(Xx[0], Xx[1], c=Yx[0], cmap="bwr", s=150, edgecolor="k")
    a.set_title(t)
plt.tight_layout(); plt.show()

##Example 4: Effect of the learning rate and the cost function (two moons)

Two-moons data, network 2-16-1 with a tanh hidden layer and a sigmoid output, mini-batches of 32.
- Learning rate η: too small learns slowly; too large makes the updates overshoot, so the cost jumps around instead of decreasing steadily.
- Cost function: MSE vs cross-entropy, with the same η and the same initialization.

In [ ]:
Xm, ym = make_moons(n_samples=600, noise=0.2, random_state=0)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=0)
Xtr, Xte = Xm_tr.T, Xm_te.T
Ytr = ym_tr[None, :].astype(float)
sizes_m = [2, 16, 1]

fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for eta in [0.01, 0.1, 0.5, 5.0]:
    p, hst = train(sizes_m, Xtr, Ytr, "tanh", "sigmoid", "ce", eta=eta, epochs=200, batch=32)
    print(f"η = {eta:<5} final cost = {hst[-1]:.4f}   test accuracy = {np.mean(predict(p, Xte, 'tanh', 'sigmoid') == ym_te):.3f}")
    ax[0].semilogy(hst, label=f"η = {eta}")
ax[0].set_xlabel("epoch"); ax[0].set_title("Cross-entropy cost vs learning rate (log scale)"); ax[0].legend()

for kind in ["mse", "ce"]:
    p, hst = train(sizes_m, Xtr, Ytr, "tanh", "sigmoid", kind, eta=0.1, epochs=200, batch=32)
    print(f"{kind.upper():4s} (η = 0.1) test accuracy = {np.mean(predict(p, Xte, 'tanh', 'sigmoid') == ym_te):.3f}")
    ax[1].semilogy(hst / hst[0], label=kind.upper())
    if kind == "ce":
        p_best = p
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("cost / initial cost"); ax[1].set_title("MSE vs cross-entropy (η = 0.1)"); ax[1].legend()

g1 = np.linspace(-2, 3, 250); g2 = np.linspace(-1.5, 2, 250)
G1, G2 = np.meshgrid(g1, g2)
P = forward(p_best, np.vstack([G1.ravel(), G2.ravel()]), "tanh", "sigmoid")[1][-1].reshape(G1.shape)
ax[2].contourf(G1, G2, P, levels=20, cmap="RdBu_r", alpha=0.6)
ax[2].contour(G1, G2, P, levels=[0.5], colors="k")
ax[2].scatter(Xm_te[:, 0], Xm_te[:, 1], c=ym_te, cmap="bwr", s=12, edgecolor="k", lw=0.3)
ax[2].set_title("Learned decision boundary (test points)")
plt.tight_layout(); plt.show()

##Example 5: Handwritten digits (softmax + cross-entropy)

1,797 images of handwritten digits, each 8 × 8 pixels, so the input is a vector in R⁶⁴. There are 10 classes. Labels are one-hot vectors y ∈ R¹⁰ (digit 3 → y = e₃). Network 64-32-10: ReLU hidden layer, softmax output, cross-entropy cost, so δᴸ = aᴸ − y.

In [ ]:
digits = load_digits()
Xd = digits.data / 16.0
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, digits.target, test_size=0.25, random_state=0, stratify=digits.target)
Yd_tr = np.eye(10)[yd_tr].T

pd_, hd = train([64, 32, 10], Xd_tr.T, Yd_tr, "relu", "softmax", "ce", eta=0.1, epochs=100, batch=32)
pred = predict(pd_, Xd_te.T, "relu", "softmax")
conf = np.zeros((10, 10), int)
np.add.at(conf, (yd_te, pred), 1)
print(f"train accuracy = {np.mean(predict(pd_, Xd_tr.T, 'relu', 'softmax') == yd_tr):.3f}")
print(f"test accuracy  = {np.mean(pred == yd_te):.3f}")

fig = plt.figure(figsize=(16, 4))
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(hd); ax1.set_xlabel("epoch"); ax1.set_title("Training cost")
ax2 = fig.add_subplot(1, 3, 2)
im = ax2.imshow(conf, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax2.text(j, i, conf[i, j], ha="center", va="center", fontsize=7)
ax2.set_xlabel("predicted"); ax2.set_ylabel("true"); ax2.set_title("Test confusion matrix"); ax2.grid(False)
ax2.set_xticks(range(10)); ax2.set_yticks(range(10))
for i, k in enumerate(rng.choice(len(yd_te), 8, replace=False)):
    a = fig.add_subplot(2, 12, 9 + i + (i // 4) * 8)
    a.imshow(Xd_te[k].reshape(8, 8), cmap="gray_r"); a.axis("off")
    a.set_title(f"pred {pred[k]}", fontsize=8, color="g" if pred[k] == yd_te[k] else "r")
plt.tight_layout(); plt.show()

#Summary

-Network: zˡ = Wˡaˡ⁻¹ + bˡ, aˡ = σ(zˡ). The activation must be nonlinear, or the layers collapse into one linear map.
-Cost: MSE or cross-entropy. Cross-entropy with a sigmoid or softmax output gives δᴸ = aᴸ − y and avoids slow learning when the output saturates.
-Backpropagation: δᴸ = ∇ₐC ⊙ σ'(zᴸ), δˡ = (Wˡ⁺¹)ᵀδˡ⁺¹ ⊙ σ'(zˡ), ∂C/∂bˡ = δˡ, ∂C/∂Wˡ = δˡ(aˡ⁻¹)ᵀ. Checked against finite differences.
-Algorithm: forward pass, backward pass, gradient step on each mini-batch. The cost is non-convex, and η controls speed and stability.